# Error Analysis

This notebook describes where the frozen final attendance model succeeds and where its predictions remain inaccurate.

## 1. Purpose and Scope

The compact Baseline + Member Reliability feature set, Random Forest model, preprocessing, and hyperparameters were fixed before the test period was evaluated. Model development is closed. The final test predictions are used here only to describe error patterns and compare the frozen model with the current-booking-count heuristic; no finding in this notebook is used to revise the model.

## 2. Load Final Test Predictions

Notebook 04 stores one privacy-preserving row per final test prediction. The artifact contains class identifiers, the target, current booking count, selected context variables, and the frozen model prediction, but no member identifiers or member lists.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PREDICTIONS_PATH = (
    Path("..") / "data" / "processed" / "final_test_predictions.parquet"
)
predictions = pd.read_parquet(PREDICTIONS_PATH)

prediction_identifier_columns = [
    "studio",
    "course",
    "class_start",
    "prediction_horizon",
]
required_columns = prediction_identifier_columns + [
    "final_attendance_count",
    "attendance_count",
    "final_model_prediction",
]
missing_columns = sorted(set(required_columns) - set(predictions.columns))
if missing_columns:
    raise ValueError(f"Missing prediction columns: {missing_columns}")
if predictions[["final_attendance_count", "final_model_prediction"]].isna().any().any():
    raise ValueError("Targets and final predictions must not be missing.")
if predictions.duplicated(prediction_identifier_columns).any():
    raise ValueError("Final prediction identifiers must be unique.")

predictions.shape

In [ ]:
predictions = predictions.copy()
predictions["error"] = (
    predictions["final_model_prediction"]
    - predictions["final_attendance_count"]
)
predictions["absolute_error"] = predictions["error"].abs()
predictions["booking_baseline_error"] = (
    predictions["attendance_count"]
    - predictions["final_attendance_count"]
)
predictions["booking_baseline_absolute_error"] = predictions[
    "booking_baseline_error"
].abs()
predictions["improvement_vs_booking_baseline"] = (
    predictions["booking_baseline_absolute_error"]
    - predictions["absolute_error"]
)

predictions.head()

## 3. Overall Error Distribution

Signed error is prediction minus actual attendance: positive values are overpredictions and negative values are underpredictions. The distribution shows whether errors are centered and whether a small number of classes account for the largest misses.

In [ ]:
error_summary = pd.Series(
    {
        "mean_error": predictions["error"].mean(),
        "median_error": predictions["error"].median(),
        "MAE": predictions["absolute_error"].mean(),
        "90th_percentile_absolute_error": predictions["absolute_error"].quantile(0.9),
        "maximum_absolute_error": predictions["absolute_error"].max(),
    },
    name="value",
)
error_summary.round(3).to_frame()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(predictions["error"], bins=15, color="#315f72", alpha=0.9)
ax.axvline(0, color="#333333", linewidth=1)
ax.set(xlabel="Prediction error (attendees)", ylabel="Number of classes")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()

Errors are close to centered but retain slight underprediction: mean error is -0.095 and median error is -0.223. Mean absolute error is 1.546 attendees. Ninety percent of classes have an absolute error below 2.956, while the maximum miss is 4.380, indicating that most errors are moderate with a small tail of larger misses.

## 4. Error by Attendance Level

Actual attendance is divided into three broad groups with adequate sample sizes: low (1–4 attendees), typical (5–7), and high (8–10). This asks whether unusually small or large classes are more difficult without creating many arbitrary categories.

In [ ]:
predictions["attendance_group"] = pd.cut(
    predictions["final_attendance_count"],
    bins=[-float("inf"), 4, 7, float("inf")],
    labels=["Low (1-4)", "Typical (5-7)", "High (8+)"],
)
attendance_level_summary = (
    predictions.groupby("attendance_group", observed=False)
    .agg(
        classes=("absolute_error", "size"),
        MAE=("absolute_error", "mean"),
        mean_error=("error", "mean"),
    )
)
attendance_level_summary.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.scatter(
    predictions["final_attendance_count"],
    predictions["absolute_error"],
    color="#315f72",
    alpha=0.65,
)
ax.set(
    xlabel="Actual final attendance",
    ylabel="Absolute prediction error",
)
ax.grid(alpha=0.3)
plt.tight_layout()

Classes with 5–7 attendees are predicted most accurately, with MAE 1.278 across 48 cases. Low-attendance classes have MAE 1.587 and are overpredicted by 1.169 attendees on average. High-attendance classes are most difficult, with MAE 1.923 and mean underprediction of 1.866 across 29 cases. The opposing signed errors indicate a tendency toward middle-sized predictions rather than a uniform bias across attendance levels.

## 5. Model vs. Current-Booking Benchmark

Positive improvement means that the final model has lower absolute error than the current-booking-count heuristic for that class. The comparison also checks whether the model's stronger RMSE mainly comes from correcting a small number of large heuristic errors.

In [ ]:
large_booking_error_threshold = predictions[
    "booking_baseline_absolute_error"
].quantile(0.9)
large_booking_error_mask = (
    predictions["booking_baseline_absolute_error"]
    >= large_booking_error_threshold
)
comparison_summary = pd.Series(
    {
        "model_better_classes": (predictions["improvement_vs_booking_baseline"] > 0).sum(),
        "model_better_share": (predictions["improvement_vs_booking_baseline"] > 0).mean(),
        "booking_baseline_better_classes": (predictions["improvement_vs_booking_baseline"] < 0).sum(),
        "booking_baseline_better_share": (predictions["improvement_vs_booking_baseline"] < 0).mean(),
        "ties": (predictions["improvement_vs_booking_baseline"] == 0).sum(),
        "mean_improvement": predictions["improvement_vs_booking_baseline"].mean(),
        "median_improvement": predictions["improvement_vs_booking_baseline"].median(),
        "largest_improvement": predictions["improvement_vs_booking_baseline"].max(),
        "largest_booking_baseline_advantage": predictions["improvement_vs_booking_baseline"].min(),
        "mean_improvement_largest_10pct_booking_errors": predictions.loc[large_booking_error_mask, "improvement_vs_booking_baseline"].mean(),
        "mean_improvement_remaining_classes": predictions.loc[~large_booking_error_mask, "improvement_vs_booking_baseline"].mean(),
    },
    name="value",
)
comparison_summary.round(3).to_frame()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(
    predictions["improvement_vs_booking_baseline"],
    bins=15,
    color="#2f6f5e",
    alpha=0.9,
)
ax.axvline(0, color="#333333", linewidth=1)
ax.set(
    xlabel="Absolute-error improvement vs booking baseline",
    ylabel="Number of classes",
)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()

The final model has lower absolute error for 69 of 125 classes (55.2%), while the booking-count heuristic is better for 56 (44.8%). Mean and median improvements are modest at 0.134 and 0.120 attendees. The advantage is concentrated in the heuristic's largest misses: for classes at or above the 90th percentile of booking-count absolute error, the model improves absolute error by 2.194 attendees on average; across the remaining classes it is worse by 0.147. This pattern is consistent with the stronger RMSE improvement being driven primarily by corrections of a smaller number of large booking-count errors. 

## 6. Largest Prediction Errors

The ten largest absolute errors are inspected as individual cases. The table includes only existing context variables that help describe the booking state and available reliability history; it does not assign causes to these misses.

In [ ]:
largest_error_columns = [
    "class_start",
    "studio",
    "course",
    "final_attendance_count",
    "attendance_count",
    "final_model_prediction",
    "absolute_error",
    "booking_baseline_absolute_error",
    "improvement_vs_booking_baseline",
    "occupancy_rate",
    "members_with_reliability_history_count_365d",
]
largest_prediction_errors = predictions.nlargest(
    10, "absolute_error"
)[largest_error_columns]
largest_prediction_errors.round(
    {
        "final_model_prediction": 3,
        "absolute_error": 3,
        "booking_baseline_absolute_error": 3,
        "improvement_vs_booking_baseline": 3,
        "occupancy_rate": 3,
    }
)

The largest misses occur across both studios, a wide occupancy range, and actual attendance from 2 to 10; no single operational context explains all ten cases. Their direction follows the attendance-level pattern: several low-attendance classes are substantially overpredicted, while several classes with 8–10 attendees are underpredicted. In some cases the model meaningfully corrects a large booking-count error, but in others the booking count already equals or nearly equals final attendance and the model moves the prediction away from that accurate baseline.

## 7. Selected Subgroup Analysis

Two dimensions were selected before examining their error results. Studio tests a central operational grouping with sufficient observations in both locations. Reliability-history coverage is represented by the number of currently booked members with 365-day reliability history, divided into three pre-specified groups with at least 39 classes each. Weekday is not added because Monday has fewer than ten test classes and the number of subgroup comparisons is deliberately limited.

In [ ]:
studio_error_summary = predictions.groupby("studio").agg(
    classes=("absolute_error", "size"),
    MAE=("absolute_error", "mean"),
    mean_error=("error", "mean"),
)

predictions["reliability_history_group"] = pd.cut(
    predictions["members_with_reliability_history_count_365d"],
    bins=[0, 3, 7, float("inf")],
    labels=["Low (1-3)", "Medium (4-7)", "High (8-11)"],
)
reliability_error_summary = (
    predictions.groupby("reliability_history_group", observed=False)
    .agg(
        classes=("absolute_error", "size"),
        MAE=("absolute_error", "mean"),
        mean_error=("error", "mean"),
    )
)

subgroup_error_summary = pd.concat(
    {
        "Studio": studio_error_summary,
        "365d reliability-history count": reliability_error_summary,
    },
    names=["dimension", "group"],
)
subgroup_error_summary.round(3)

Studio differences are modest: MAE is 1.498 for Cb and 1.610 for Nk, with Nk showing somewhat stronger underprediction (-0.207). Error does not decrease as the 365-day reliability-history count rises. The low, medium, and high groups have MAE 1.461, 1.509, and 1.681 respectively, and the high-count group is underpredicted by 0.523 on average. This count also increases with the number of booked members and therefore overlaps with class size; the pattern cannot be attributed to reliability-history availability alone.

## 8. Main Findings and Limitations

The frozen model's errors are broadly centered, with slight average underprediction and a limited tail of larger misses. Middle-sized classes are predicted most accurately; low-attendance classes tend to be overpredicted and high-attendance classes underpredicted. The model improves on current booking count for a small majority of classes, but its main benefit is correcting the heuristic's largest errors, which explains the clearer RMSE gain. The largest model misses include both successful corrections and cases where an already accurate booking count would have been preferable.

The two pre-selected subgroup views show only a modest studio difference and no decrease in error at higher reliability-history counts. These comparisons are limited by 125 test classes, correlated operational variables, and the fact that reliability-history count also reflects the number of booked members. They are descriptive rather than causal or statistically confirmed. 